# 🤖 RAG-Powered Customer Support Chatbot

## Learn to build a chatbot that answers customer queries using company policy documents

### What you'll learn:
1. **Document Loading** - Load and process PDF/text documents
2. **Text Splitting** - Chunk documents for better retrieval
3. **Embeddings** - Convert text to vectors
4. **Vector Store** - Store and search embeddings efficiently
5. **RAG Chain** - Combine retrieval with LLM for accurate answers

### Tech Stack:
- **LLM**: Groq (Llama 3.1 70B) - Fast & Free
- **Embeddings**: HuggingFace (all-MiniLM-L6-v2) - Free & Local
- **Vector Store**: ChromaDB - Free & Local
- **Framework**: LangChain

---

## Step 0: Install Required Packages

Run this cell first to install all dependencies.

In [ ]:
# Install required packages
!pip install langchain langchain-groq langchain-community langchain-huggingface langchain-chroma langchain-text-splitters chromadb pypdf sentence-transformers -q

## Step 1: Import Libraries & Setup API Key

We'll import all necessary libraries and configure our Groq API key.

In [1]:
# Core imports
import os
from getpass import getpass

# LangChain imports
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [2]:
# Setup Groq API Key
# Option 1: Enter when prompted (more secure)
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
print(f"{GROQ_API_KEY} is set from environment variable.")

# Option 2: Directly set (less secure, don't commit to git)
# os.environ["GROQ_API_KEY"] = "your-api-key-here"

print("✅ API Key configured!")

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 is set from environment variable.
✅ API Key configured!


## Step 2: Load the Document

We'll load our company policy document. LangChain supports various loaders:
- `TextLoader` - For .txt and .md files
- `PyPDFLoader` - For PDF files
- `CSVLoader` - For CSV files
- And many more!

For this example, we're using the ShopEasy policy markdown file.

In [3]:
# Load the document
# Using TextLoader for markdown file (also works with PDF using PyPDFLoader)

loader = TextLoader("shopeasy_policy.md", encoding="utf-8")
documents = loader.load()

# If you have a PDF file, use this instead:
# loader = PyPDFLoader("shopeasy_policy.pdf")
# documents = loader.load()

print(f"✅ Loaded {len(documents)} document(s)")
print(f"📄 Document length: {len(documents[0].page_content)} characters")
print(f"\n--- First 500 characters ---")
print(documents[0].page_content[:500])

✅ Loaded 1 document(s)
📄 Document length: 9236 characters

--- First 500 characters ---
# ShopEasy E-Commerce - Company Policies

## About ShopEasy

ShopEasy is India's leading online marketplace founded in 2018, headquartered in Bangalore. We serve over 10 million customers across 500+ cities in India. Our mission is to provide quality products at affordable prices with exceptional customer service.

---

## 1. TERMS AND CONDITIONS

### 1.1 Account Registration
- Users must be at least 18 years old to create an account
- One account per person/email address is permitted
- Users ar


## Step 3: Split Document into Chunks

### Why do we chunk?
1. **LLM Context Limits** - LLMs have maximum input sizes
2. **Better Retrieval** - Smaller chunks = more precise matching
3. **Cost Efficiency** - Send only relevant parts to LLM

### Key Parameters:
- `chunk_size` - Maximum characters per chunk (start with 500-1000)
- `chunk_overlap` - Overlapping characters between chunks (prevents cutting off context)

In [4]:
# Create text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # Max characters per chunk
    chunk_overlap=200,    # Overlap between chunks
    length_function=len,
    separators=["\n\n", "\n", ".", " ", ""]  # Split priority
)

# Split the documents
chunks = text_splitter.split_documents(documents)

print(f"✅ Document split into {len(chunks)} chunks")
print(f"\n--- Example Chunk (Chunk #5) ---")
print(f"Content: {chunks[5].page_content[:300]}...")
print(f"\nMetadata: {chunks[5].metadata}")

✅ Document split into 11 chunks

--- Example Chunk (Chunk #5) ---
Content: **Special Items:**
- Large appliances: ₹199 - ₹499 (includes installation)
- Furniture: ₹299 - ₹999 (based on size and location)

### 3.3 Delivery Process
1. Order confirmation email sent immediately
2. Dispatch notification with tracking number within 24-48 hours
3. Real-time tracking available on ...

Metadata: {'source': 'shopeasy_policy.md'}


In [5]:
# Let's visualize chunk sizes
chunk_lengths = [len(chunk.page_content) for chunk in chunks]

print("📊 Chunk Statistics:")
print(f"   Total chunks: {len(chunks)}")
print(f"   Min chunk size: {min(chunk_lengths)} chars")
print(f"   Max chunk size: {max(chunk_lengths)} chars")
print(f"   Avg chunk size: {sum(chunk_lengths)//len(chunk_lengths)} chars")

📊 Chunk Statistics:
   Total chunks: 11
   Min chunk size: 730 chars
   Max chunk size: 995 chars
   Avg chunk size: 911 chars


## Step 4: Create Embeddings

### What are Embeddings?
Embeddings convert text into numerical vectors (arrays of numbers) that capture semantic meaning.

**Example:**
- "Happy" → [0.2, 0.8, 0.1, ...]
- "Joyful" → [0.21, 0.79, 0.11, ...] (similar to "Happy")
- "Car" → [0.9, 0.1, 0.7, ...] (different from "Happy")

We're using **HuggingFace's all-MiniLM-L6-v2** model:
- Free & runs locally
- Fast & lightweight
- 384-dimensional vectors

In [6]:
# Initialize embedding model
# This will download the model on first run (~90MB)

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'},  # Use 'cuda' if you have GPU
    encode_kwargs={'normalize_embeddings': True}
)

print("✅ Embedding model loaded!")

✅ Embedding model loaded!


In [7]:
# Let's see what an embedding looks like
sample_text = "What is the return policy?"
sample_embedding = embeddings.embed_query(sample_text)

print(f"📝 Text: '{sample_text}'")
print(f"🔢 Embedding dimension: {len(sample_embedding)}")
print(f"🔢 First 10 values: {sample_embedding[:10]}")

📝 Text: 'What is the return policy?'
🔢 Embedding dimension: 384
🔢 First 10 values: [-0.03421613201498985, 0.06547138839960098, 0.026852790266275406, 0.01352331880480051, 0.0115633774548769, 0.04131702706217766, 0.025687813758850098, -0.05009191110730171, -0.0492679737508297, 0.028764285147190094]


## Step 5: Create Vector Store

### What is a Vector Store?
A vector store (or vector database) stores embeddings and enables fast similarity search.

When you ask a question:
1. Question → Embedding
2. Search vector store for similar embeddings
3. Return the most similar document chunks

We're using **ChromaDB**:
- Free & open-source
- Runs locally (no setup needed)
- Great for learning and small-medium projects

In [8]:
# Create vector store from chunks
# This embeds all chunks and stores them

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,  # Changed from embedding_function to embedding
    persist_directory="./chroma_db",  # Save to disk
    collection_name="shopeasy_policies"
)

print(f"✅ Vector store created with {vectorstore._collection.count()} vectors!")

✅ Vector store created with 22 vectors!


In [9]:
# Test the retriever - let's search for relevant chunks
query = "What is the return policy for electronics?"

# Search for similar documents
similar_docs = vectorstore.similarity_search(query, k=3)  # Get top 3 matches

print(f"🔍 Query: '{query}'\n")
print("📄 Top 3 relevant chunks found:")
print("=" * 50)

for i, doc in enumerate(similar_docs, 1):
    print(f"\n--- Chunk {i} ---")
    print(doc.page_content[:400])
    print("...")

🔍 Query: 'What is the return policy for electronics?'

📄 Top 3 relevant chunks found:

--- Chunk 1 ---
---

## 4. RETURN AND REFUND POLICY

### 4.1 Return Window

| Product Category | Return Window |
|-----------------|---------------|
| Electronics | 10 days |
| Fashion & Apparel | 15 days |
| Books & Stationery | 10 days |
| Home & Kitchen | 15 days |
| Groceries & Food | Non-returnable |
| Personal Care & Hygiene | Non-returnable |
| Customized Products | Non-returnable |
| Innerwear & Swimwear 
...

--- Chunk 2 ---
---

## 4. RETURN AND REFUND POLICY

### 4.1 Return Window

| Product Category | Return Window |
|-----------------|---------------|
| Electronics | 10 days |
| Fashion & Apparel | 15 days |
| Books & Stationery | 10 days |
| Home & Kitchen | 15 days |
| Groceries & Food | Non-returnable |
| Personal Care & Hygiene | Non-returnable |
| Customized Products | Non-returnable |
| Innerwear & Swimwear 
...

--- Chunk 3 ---
### 4.4 Refund Methods and Timelines

| Payment Metho

## Step 6: Initialize the LLM (Groq)

We're using **Groq** with **Llama 3.1 70B**:
- Extremely fast inference
- Free tier available
- High-quality responses

Other available models on Groq:
- `llama-3.1-70b-versatile` (recommended)
- `llama-3.1-8b-instant` (faster, less capable)
- `mixtral-8x7b-32768` (good alternative)

In [10]:
# Initialize Groq LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.3,  # Lower = more focused, Higher = more creative
    max_tokens=1024,
)

# Quick test
response = llm.invoke("Say 'Hello, I am ready to help!' in one line.")
print(f"✅ LLM initialized!")
print(f"🤖 LLM says: {response.content}")

✅ LLM initialized!
🤖 LLM says: Hello, I am ready to help!


## Step 7: Create the RAG Chain

Now we combine everything:
1. **Retriever** - Fetches relevant chunks from vector store
2. **Prompt Template** - Instructs the LLM how to use the context
3. **LLM** - Generates the final answer

This is the **heart of RAG**!

In [11]:
# Create a custom prompt template
# This tells the LLM how to behave and use the context

prompt_template = """You are a helpful customer support assistant for ShopEasy, an e-commerce company.
Use the following context from our policy documents to answer the customer's question.

Guidelines:
- Be friendly, professional, and helpful
- Only answer based on the provided context
- If the answer is not in the context, say "I don't have information about that in our policy documents. Please contact our support team at 1800-123-4567."
- Keep answers concise but complete
- If relevant, mention specific timelines, amounts, or conditions

Context from policy documents:
{context}

Customer Question: {question}

Helpful Answer:"""

PROMPT = ChatPromptTemplate.from_template(prompt_template)

print("✅ Prompt template created!")

✅ Prompt template created!


In [12]:
# Create the retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}  # Retrieve top 4 chunks
)

# Helper function to format retrieved documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Create the RAG chain using LCEL (LangChain Expression Language)
# This is the modern, recommended approach
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | PROMPT
    | llm
    | StrOutputParser()
)

print("✅ RAG Chain created! Ready to answer questions.")

✅ RAG Chain created! Ready to answer questions.


## Step 8: Test the Chatbot! 🎉

Let's test our RAG chatbot with various customer queries.

In [ ]:
# Helper function to ask questions
def ask_question(question: str):
    """Ask a question and get an answer with sources."""
    print(f"\n{'='*60}")
    print(f"❓ Customer: {question}")
    print(f"{'='*60}")
    
    # Get response using the new chain
    response = rag_chain.invoke(question)
    
    print(f"\n🤖 ShopEasy Assistant:\n{response}")
    
    # Get source documents separately for debugging
    # source_docs = retriever.invoke(question)
    # print(f"\n📚 Sources used: {len(source_docs)} chunks")
    
    return response

In [ ]:
ask_question("If my order get cancelled, then what will be my order status?")

## Step 9: Interactive Chat Mode 💬

Let's create an interactive chat interface where you can ask any question!

In [ ]:
# Interactive chat loop
print("="*60)
print("🛒 Welcome to ShopEasy Customer Support!")
print("="*60)
print("Ask any question about our policies.")
print("Type 'quit' or 'exit' to end the chat.\n")

while True:
    # Get user input
    user_question = input("\n👤 You: ").strip()
    
    # Check for exit
    if user_question.lower() in ['quit', 'exit', 'q']:
        print("\n👋 Thank you for chatting with ShopEasy! Have a great day!")
        break
    
    # Skip empty input
    if not user_question:
        print("Please enter a question.")
        continue
    
    # Get and display response
    try:
        response = rag_chain.invoke(user_question)
        print(f"\n🤖 Assistant: {response}")
    except Exception as e:
        print(f"\n❌ Error: {e}")

---

## 🎓 What You've Learned

Congratulations! You've built a complete RAG-powered customer support chatbot! Here's what you learned:

| Step | Concept | Package/Tool Used |
|------|---------|-------------------|
| 1 | Document Loading | `TextLoader` (langchain_community) |
| 2 | Text Chunking | `RecursiveCharacterTextSplitter` (langchain_text_splitters) |
| 3 | Embeddings | `HuggingFaceEmbeddings` (langchain_huggingface) |
| 4 | Vector Storage | `Chroma` (langchain_chroma) |
| 5 | Retrieval | `as_retriever()` with similarity search |
| 6 | LLM Integration | `ChatGroq` - Llama 3.3 70B (langchain_groq) |
| 7 | Prompt Engineering | `ChatPromptTemplate` (langchain_core) |
| 8 | RAG Chain | LCEL with `RunnablePassthrough` & `StrOutputParser` |

### Key Concepts Mastered:

**1. RAG Pipeline Flow:**
```
Document → Chunks → Embeddings → Vector Store → Retriever → LLM → Answer
```

**2. LCEL (LangChain Expression Language):**
```python
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | PROMPT
    | llm
    | StrOutputParser()
)
```
This modern approach uses the `|` pipe operator to chain components - more flexible and future-proof than the deprecated `RetrievalQA`.

**3. Modern LangChain Package Structure:**
| Package | Purpose |
|---------|---------|
| `langchain-core` | Base abstractions (prompts, runnables, output parsers) |
| `langchain-text-splitters` | Text chunking utilities |
| `langchain-chroma` | ChromaDB vector store integration |
| `langchain-groq` | Groq LLM integration |
| `langchain-huggingface` | HuggingFace embeddings |
| `langchain-community` | Community integrations (document loaders) |

---

## 🚀 Next Steps to Level Up

### Beginner:
- [ ] Try different chunk sizes (500, 1500) and see how it affects answers
- [ ] Add more policy documents to the vector store
- [ ] Try different LLM models on Groq (`llama-3.1-8b-instant`, `mixtral-8x7b-32768`)

### Intermediate:
- [ ] Implement conversation memory (remember previous questions)
- [ ] Add source citations in responses
- [ ] Try hybrid search (keyword + semantic)
- [ ] Implement re-ranking for better retrieval

### Advanced:
- [ ] Build a web UI with Streamlit or Gradio
- [ ] Implement multi-query retrieval
- [ ] Add evaluation metrics (RAGAS)
- [ ] Deploy as an API
- [ ] Explore LangGraph for complex multi-step workflows

---

## 📚 Bonus: Understanding the RAG Pipeline Visually

```
                         RAG PIPELINE
                         
    ┌─────────────────────────────────────────────────────────┐
    │                    INDEXING PHASE                        │
    │                  (Done once, stored)                     │
    │                                                          │
    │   PDF/Doc  ──▶  Chunks  ──▶  Embeddings  ──▶  VectorDB  │
    │                                                          │
    └─────────────────────────────────────────────────────────┘
                              │
                              │ Stored
                              ▼
    ┌─────────────────────────────────────────────────────────┐
    │                    QUERY PHASE                           │
    │               (Done for each question)                   │
    │                                                          │
    │   Question  ──▶  Embedding  ──▶  Similar Search         │
    │                                        │                 │
    │                                        ▼                 │
    │                              Relevant Chunks             │
    │                                        │                 │
    │                                        ▼                 │
    │                     Question + Chunks + Prompt           │
    │                                        │                 │
    │                                        ▼                 │
    │                                      LLM                 │
    │                                        │                 │
    │                                        ▼                 │
    │                                    Answer                │
    │                                                          │
    └─────────────────────────────────────────────────────────┘
```

## 🧹 Cleanup (Optional)

Run this to delete the vector store if you want to start fresh.

In [ ]:
# Optional: Delete the vector store to start fresh
# import shutil
# shutil.rmtree("./chroma_db", ignore_errors=True)
# print("✅ Vector store deleted!")